<a href="https://colab.research.google.com/github/javierraneros-cell/Mineria_de_Datos-Actividad_Practica_UD3/blob/main/Actividad_Pr%C3%A1ctica_UD3_Miner%C3%ADa_de_Datos_e_IA_Corporativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TÉCNICAS DE MINERÍA DE DATOS (I) - UD3 MINERIA DE DATOS E IA CORPORATIVA


## INTRODUCCIÓN

Una bodega vallisoletana produce vinos de tres calidades: joven, crianza y reserva. Un día ocurre un problema en el etiquetado de las botellas, resultando en la mezcla de botellas de distintas calidades. Para abordar esta situación se emplean los datos disponibles del día anterior (vino_entrenamiento.csv) que consisten en 13 variables cuantitativas (alcohol, intensidad del color, etc.) y su calidad correspondiente. Para las botellas afectadas por el fallo se miden las mismas 13 variable cualitativas y se almacenan en otro conjunto de datos (vino_prueba.csv). El objetivo es desarrollar un modelo de clasificación que utilice la información de las 13 variables cuantitativas para reetiquetar las botellas intentando cometer el menor número de errores. El motivo es obvio, si se vende vino reserva a precio de joven la bodega pierde dinero, mientras que si vende vino joven como si fuera reserva la bodega corre el peligro de perder clientes. Días después, revisando los registros de la máquina de etiquetado se consigue conocer de forma inequívoca de qué tipo era cada botella y se añade al conjunto de datos vino_prueba.csv. Por lo tanto, se podrá comparar los resultados arrojados por el modelo creado con los valores reales.

En esta actividad se pretende aplicar un modelo **Naïve-Bayes** para tratar de clasificar las botellas. Además, se va a utilizar el análisis de componentes principales para mejorar el rendimiento del primer modelo.

# **Problema 1: Clasificación**

## Descargar los datos del aula e impórtalos en Python.

Para este ejercicio se utilizan dos ficheros:

*   vino_entrenamiento.csv
*   vino_prueba.csv

In [1]:
# Instalar gdown si no está instalado
!pip install gdown


Para incorporar el fichero de Google Drive al proyecto hay que descargarlo con gdown y guaradarlos como ficheros CSV para despues ser usados en la creación del/los Datasets.

El listado de ficheros que se encuentran en la carpeta /datos https://drive.google.com/drive/folders/1-Yi2JSMfZOx5wcBs7u1MkiII5Hcuh2xU?usp=drive_link es:

*   vino_entrenamiento.csv --> 1ywxVg0p2DyslW5AH0hPoS6VCBvlsvbM6
*   vino_prueba.csv --> 1ndmj7qYoYE_CHnNwr2399wYkso42XfCy

In [2]:
import gdown

#IDs de los archivos a descargar de Google Drive que tienen que estar compartidos
# vino_entrenamiento.csv --> 1ywxVg0p2DyslW5AH0hPoS6VCBvlsvbM6
# vino_prueba.csv --> 1ndmj7qYoYE_CHnNwr2399wYkso42XfCy

file_vino_entrenamiento_id = '1ywxVg0p2DyslW5AH0hPoS6VCBvlsvbM6'
file_vino_prueba_id = '1ndmj7qYoYE_CHnNwr2399wYkso42XfCy'

output_vino_entrenamiento = 'vino_entrenamiento.csv'
output_vino_prueba = 'vino_prueba.csv'

#FICHERO VINO ENTRENAMIENTO
url = f'https://drive.google.com/uc?id={file_vino_entrenamiento_id}'
gdown.download(url, output_vino_entrenamiento, quiet=False)
print('Fichero vino entrenamiento descargado...')

#FICHERO VINO PRUEBA
url = f'https://drive.google.com/uc?id={file_vino_prueba_id}'
gdown.download(url, output_vino_prueba, quiet=False)
print('Fichero vino prueba descargado...')

Downloading...
From: https://drive.google.com/uc?id=1ywxVg0p2DyslW5AH0hPoS6VCBvlsvbM6
To: /content/vino_entrenamiento.csv
100%|██████████| 9.99k/9.99k [00:00<00:00, 19.1MB/s]


Fichero vino entrenamiento descargado...


Downloading...
From: https://drive.google.com/uc?id=1ndmj7qYoYE_CHnNwr2399wYkso42XfCy
To: /content/vino_prueba.csv
100%|██████████| 2.68k/2.68k [00:00<00:00, 7.99MB/s]

Fichero vino prueba descargado...


## Creación Datasets

Crearemos un dataset por cada fichero CSV para ser usado en el modelo mas adelante.



In [3]:
import pandas as pd

#Creación de dataset para analisis de vino y vinos pruebas

df_vino_entrenamiento = pd.read_csv('vino_entrenamiento.csv', sep=',')
print(f"Archivo `{output_vino_entrenamiento}` cargado exitosamente...")
display(df_vino_entrenamiento.head())

df_vino_prueba = pd.read_csv('vino_prueba.csv', sep=',')
print(f"Archivo '{output_vino_prueba}' cargado exitosamente")
display(df_vino_prueba.head())

Archivo `vino_entrenamiento.csv` cargado exitosamente...


,class,Alcohol,Malic acid,Ash,Alcalinity of ash,Magnesium,Total phenols,Flavanoids,Nonflavanoid phenols,Proanthocyanins,Color intensity,Hue,OD280/OD315 of diluted wines,Proline
0,Crianza,12.47,1.52,2.20,19.0,162,2.50,2.27,0.32,3.28,2.60,1.16,2.63,937
1,Crianza,12.00,1.51,2.42,22.0,86,1.45,1.25,0.50,1.63,3.60,1.05,2.65,450
2,Joven,13.50,1.81,2.61,20.0,96,2.53,2.61,0.28,1.66,3.52,1.12,3.82,845
3,Crianza,11.61,1.35,2.70,20.0,94,2.74,2.92,0.29,2.49,2.65,0.96,3.26,680
4,Crianza,11.56,2.05,3.23,28.5,119,3.18,5.08,0.47,1.87,6.00,0.93,3.69,465


Archivo 'vino_prueba.csv' cargado exitosamente


,class,Alcohol,Malic acid,Ash,Alcalinity of ash,Magnesium,Total phenols,Flavanoids,Nonflavanoid phenols,Proanthocyanins,Color intensity,Hue,OD280/OD315 of diluted wines,Proline
0,Crianza,12.21,1.19,1.75,16.8,151,1.85,1.28,0.14,2.50,2.85,1.28,3.07,718
1,Crianza,11.82,1.72,1.88,19.5,86,2.50,1.64,0.37,1.42,2.06,0.94,2.44,415
2,Joven,14.06,1.63,2.28,16.0,126,3.00,3.17,0.24,2.10,5.65,1.09,3.71,780
3,Crianza,11.82,1.47,1.99,20.8,86,1.98,1.60,0.30,1.53,1.95,0.95,3.33,495
4,Joven,14.22,1.70,2.30,16.3,118,3.20,3.00,0.26,2.03,6.38,0.94,3.31,970


## **Creación Modelo de clasificación Naïve-Bayes**

Se tiene que crear un modelo de clasificación basado en Naive-Bayes, que nos permita clasificar y determinar la probabilidad de que un determinado vino pertenezca a al grupo K (crizanza, joven, reserva) dadas las evidencias almacenadas

Ya tenemos los Datasets, ahora tendremos que cargarlo en una matriz el dataset vino_entreanmiento y en otra matriz los valores de pruebas para predecir, junto las cabeceras de los atributos que formaran el array de ***variables***

Creamos el modelo modificado expuesto en la unidad para recorra la matriz de atributos pasados por parametro

In [4]:
def naive_bayes(datos, variables, target, X_new):

    grupos = datos[target].unique()
    n = datos.shape[0]

    # -----------------------------
    # 1. Probabilidades a priori
    # -----------------------------
    priori = np.array([
        datos.loc[datos[target] == g].shape[0] / n
        for g in grupos
    ])

    # -------------------------------------------------------------
    # Precalculamos medias y desviaciones antes de la clasificación
    # -------------------------------------------------------------
    medias = {}
    desvios = {}
    for g in grupos:
        subset = datos[datos[target] == g]
        medias[g] = subset[variables].mean().values
        desvios[g] = subset[variables].std().values

    # ---------------------------------------
    # Función normal para atributos continuos
    # ---------------------------------------
    normal = lambda x, m, s: np.exp(-((x - m)**2) / (2 * s**2)) / (s * np.sqrt(2*np.pi))

    # Estimación de los P(X=x|G=k_j):
    predicciones = np.zeros(len(X_new), dtype=object)

    # -----------------------------
    # Bucle de X_new para su clasificación
    # -----------------------------
    for i, tupla in enumerate(X_new):
        verosim = np.ones(len(grupos))
        for g_index, g in enumerate(grupos):
            mu = medias[g]
            sig = desvios[g]
            # Producto de verosimilitudes (independencia condicional) por cada atributo
            for j, valor in enumerate(tupla):
                verosim[g_index] *= normal(valor, mu[j], sig[j])

        # Posterior proporcional a priori * verosimilitud
        posterior = priori * verosim

        # Clase predicha
        predicciones[i] = grupos[np.argmax(posterior)]

    return predicciones

## **Llamada la funcion Naïve-Bayes** para predecir los valores de CLASS

Una vez tenemos el modelo creado Naive-Bayes, ya podemos llamarlo con los datos de entrenamiento y los datos de pruebas.

Previamente tenemos que crear la matriz



In [5]:
import numpy as np
import pandas as pd

target = 'class'
#Quitamos el atributo class del Dataset que queremos enviar a predecir
df_X_new = df_vino_prueba.drop(columns=[target])

variables = list(df_X_new.columns)
X_new = np.array(df_X_new.values)

# Ejecutamos el modelo con los datos de pruebas para Clasificar
pred = naive_bayes(df_vino_entrenamiento, variables, target, X_new)

#Pintamos el resultado
print(pred)

['Crianza' 'Crianza' 'Joven' 'Crianza' 'Joven' 'Crianza' 'Crianza' 'Joven'
 'Reserva' 'Joven' 'Joven' 'Joven' 'Joven' 'Reserva' 'Crianza' 'Crianza'
 'Reserva' 'Crianza' 'Crianza' 'Reserva' 'Joven' 'Reserva' 'Joven' 'Joven'
 'Crianza' 'Reserva' 'Crianza' 'Reserva' 'Crianza' 'Crianza' 'Reserva'
 'Reserva' 'Reserva' 'Reserva' 'Crianza' 'Joven']


In [6]:
import pandas as pd
import numpy as np

# 1. Extraer clases reales de el dataset de las pruebas
y_real = df_vino_prueba[target].values

df_comp = pd.DataFrame({
    "Predicho": pred,
    "Real": y_real,
    "Correcto": pred == y_real
})
print(df_comp)

#Tabla con estadistica:
print(pd.crosstab(df_comp['Real'], df_comp['Predicho']))

grupos = np.unique(y_real)

# Calculate and print accuracy for each class
for g in grupos:
    mask = (df_comp['Real'] == g)
    aciertos = df_comp.loc[mask, 'Correcto'].sum()
    total = mask.sum()
    acc = aciertos / total
    print(f"Clase {g}: {aciertos}/{total} aciertos ({acc:.2%})")

# Porcentaje de acierto total
accuracy = (pred == y_real).mean()
print(f"Porcentaje de aciertos TOTALES: {accuracy:.3f}")

   Predicho     Real  Correcto
0   Crianza  Crianza      True
1   Crianza  Crianza      True
2     Joven    Joven      True
3   Crianza  Crianza      True
4     Joven    Joven      True
5   Crianza  Crianza      True
6   Crianza  Crianza      True
7     Joven    Joven      True
8   Reserva  Reserva      True
9     Joven    Joven      True
10    Joven    Joven      True
11    Joven  Crianza     False
12    Joven    Joven      True
13  Reserva  Crianza     False
14  Crianza  Crianza      True
15  Crianza  Crianza      True
16  Reserva  Crianza     False
17  Crianza  Crianza      True
18  Crianza  Crianza      True
19  Reserva  Reserva      True
20    Joven    Joven      True
21  Reserva  Reserva      True
22    Joven    Joven      True
23    Joven    Joven      True
24  Crianza  Crianza      True
25  Reserva  Reserva      True
26  Crianza  Crianza      True
27  Reserva  Reserva      True
28  Crianza  Crianza      True
29  Crianza  Crianza      True
30  Reserva  Reserva      True
31  Rese

# **Problema 2: Mejora del clasificador con ACP**

## **Análisis de Componentes Principales ACP**

Realiza un análisis de componentes principales sobre el conjunto de entrenamiento *vino_entrenamiento.csv* y calcular tanto las componentes principales (***autovectores***) como sus inercias (***autovalores***).

Queremos mantener un 80% de inercia total

## **Normalización**

Recordamos que para realizar el análisis de Componentes Principales sobre el dataset que contiene los datos de *vino_entrenamiento.csv* tenemos que normalizar primero.

En este caso como los valores representan diferentes medidas, habrá que normalizar por la estandarización y no por el sistema de min-max, transformando las variables originales de forma que pasan a ser adimensionales y se elimina la dependencia respecto de las unidades de medidas empleadas mediante Z-SCORE o tipificación, pasando a ser la media 0 y la desviación típica 1.

Se aplica a todos los valores de los atributos de el dataset porque son atributos cuantitativos y normalizaremos con StandardScaler.

In [7]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

#Quitamos el atributo class del Dataset que queremos normalizar para que solo
#sean valors cuantitativos:
target = 'class'
df_vino_entrenamiento_cuantitativo = df_vino_entrenamiento.drop(columns=[target])

scaler = StandardScaler()

df_vino_entrenamiento_normalizado = pd.DataFrame(scaler.fit_transform(df_vino_entrenamiento_cuantitativo), columns=df_vino_entrenamiento_cuantitativo.columns)
df_vino_entrenamiento_normalizado.head()

,Alcohol,Malic acid,Ash,Alcalinity of ash,Magnesium,Total phenols,Flavanoids,Nonflavanoid phenols,Proanthocyanins,Color intensity,Hue,OD280/OD315 of diluted wines,Proline
0,-0.682924,-0.761564,-0.828453,-0.259946,4.597188,0.316755,0.217194,-0.398669,3.270217,-1.067010,0.840177,0.012850,0.550618
1,-1.259354,-0.770507,0.046196,0.652220,-0.979796,-1.360979,-0.804183,1.073872,0.065919,-0.642478,0.372480,0.041815,-0.959971
2,0.580317,-0.502209,0.801575,0.044109,-0.245983,0.364691,0.557653,-0.725900,0.124179,-0.676441,0.670106,1.736244,0.265250
3,-1.737668,-0.913599,1.159386,0.044109,-0.392745,0.700238,0.868072,-0.644093,1.736038,-1.045784,-0.010180,0.925235,-0.246551
4,-1.798991,-0.287570,3.266494,2.628580,1.441789,1.403288,3.030989,0.828448,0.531998,0.376398,-0.137734,1.547974,-0.913444


In [8]:
from sklearn.decomposition import PCA
import numpy as np

# Crear una instancia de PCA
pca = PCA()

# Ajustar PCA a los datos normalizados
pca.fit(df_vino_entrenamiento_normalizado)

# Crear un Dataset para visualizar los autovectores con los nombres de las columnas
df_autovectores = pd.DataFrame(
    pca.components_.round(3),
    columns= df_vino_entrenamiento_normalizado.columns,
    index=[f'PC{i+1}' for i in range(len(pca.components_))]
)

print("\nAutovectores (Componentes Principales) con la contribución de cada atributo original:")
display(df_autovectores)


Autovectores (Componentes Principales) con la contribución de cada atributo original:


,Alcohol,Malic acid,Ash,Alcalinity of ash,Magnesium,Total phenols,Flavanoids,Nonflavanoid phenols,Proanthocyanins,Color intensity,Hue,OD280/OD315 of diluted wines,Proline
PC1,0.127,-0.269,0.025,-0.242,0.123,0.395,0.422,-0.285,0.317,-0.098,0.301,0.368,0.293
PC2,0.485,0.201,0.233,-0.085,0.338,0.064,-0.024,-0.001,0.044,0.535,-0.284,-0.202,0.367
PC3,-0.197,-0.026,0.672,0.568,0.304,0.127,0.102,0.171,0.075,-0.066,0.099,0.077,-0.109
PC4,-0.153,-0.258,-0.039,-0.043,0.592,-0.239,-0.132,-0.534,-0.403,-0.118,0.054,-0.124,0.038
PC5,0.140,-0.439,0.177,-0.199,-0.072,-0.178,-0.147,0.509,-0.259,-0.005,0.466,-0.138,0.308
PC6,0.309,0.583,0.154,-0.016,-0.074,-0.008,0.013,-0.062,-0.486,-0.365,0.173,0.331,0.148
PC7,-0.190,0.395,-0.220,-0.273,0.561,-0.085,-0.062,0.389,0.353,-0.215,0.189,-0.048,-0.003
PC8,0.471,0.044,-0.176,0.433,-0.043,-0.249,-0.173,-0.250,0.371,-0.060,0.463,-0.202,-0.073
PC9,-0.282,0.228,0.523,-0.395,-0.287,-0.293,-0.022,-0.334,0.255,0.017,0.158,-0.253,0.066
PC10,-0.204,0.241,-0.152,0.034,-0.022,0.465,0.192,-0.055,-0.297,0.360,0.474,-0.395,-0.145


Ahora, vamos a calcular la varianza explicada acumulada para determinar cuántas componentes principales necesitamos para retener al menos el 80% de la inercia total.

Se observa como resultado que con los 5 primeros Autovalores obtenemos el 80% de la inercia total

In [9]:

# Varianza explicada por cada componente
varianza_explicada_ratio = pca.explained_variance_ratio_

print('\nAutovalores (INERCIAS) - RATIO %:')
for i in range(len(pca.explained_variance_)):
  ratio_inercia = np.round(varianza_explicada_ratio[i],3)
  print(f'λ{i+1} = {np.round(pca.explained_variance_[i],3)} - {ratio_inercia:.2%}')


Autovalores (INERCIAS) - RATIO %:
λ1 = 4.818 - 36.80%
λ2 = 2.553 - 19.50%
λ3 = 1.532 - 11.70%
λ4 = 0.843 - 6.40%
λ5 = 0.736 - 5.60%
λ6 = 0.701 - 5.40%
λ7 = 0.54 - 4.10%
λ8 = 0.333 - 2.50%
λ9 = 0.284 - 2.20%
λ10 = 0.261 - 2.00%
λ11 = 0.228 - 1.70%
λ12 = 0.154 - 1.20%
λ13 = 0.111 - 0.90%


### **Clasificación Naïve-Bayes con ACP**

Ahora, utilizaremos los 5 componentes principales transformados para entrenar y evaluar el modelo Naïve-Bayes, que representan el 80% de la inercia total.

También tendremos que normalizar el dataset de vinos_pruebas, para ello primero de igual forma que eliminamos el atributo class para solo tener valores cuantitativos.

Una vez tenemos el dataset con los datos cuantitativos

In [21]:
#Normalizacion datos de pruebas
#Quitamos el atributo class del Dataset que queremos normalizar para que solo
#sean valors cuantitativos igual que en el caso de los vinos entrenamiento:
target = 'class'
df_vino_prueba_cuantitativo = df_vino_prueba.drop(columns=[target])

# Pasamos por el scaler ya inicializado y entrenado anteriormente
# para que reduzca de igual forma eliminando los datos redundantes
df_vino_prueba_normalizado_array = scaler.transform(df_vino_prueba_cuantitativo)
df_vino_prueba_normalizado = pd.DataFrame(df_vino_prueba_normalizado_array, columns=df_vino_prueba_cuantitativo.columns)

print("Dataset vino_prueba normalizado :")
df_vino_prueba_normalizado.head()

Dataset vino_prueba normalizado :


,Alcohol,Malic acid,Ash,Alcalinity of ash,Magnesium,Total phenols,Flavanoids,Nonflavanoid phenols,Proanthocyanins,Color intensity,Hue,OD280/OD315 of diluted wines,Proline
0,-1.001800,-1.056692,-2.617507,-0.928868,3.789993,-0.721842,-0.774143,-1.871210,1.755458,-0.960877,1.350391,0.650072,-0.128682
1,-1.480114,-0.582698,-2.100669,-0.107918,-0.979796,0.316755,-0.413656,0.010370,-0.341901,-1.296258,-0.095216,-0.262313,-1.068535
2,1.267127,-0.663188,-0.510398,-1.172112,1.955459,1.115677,1.118410,-1.053132,0.978658,0.227812,0.542552,1.576939,0.063631
3,-1.480114,-0.806280,-1.663345,0.287354,-0.979796,-0.514123,-0.453710,-0.562285,-0.128281,-1.342956,-0.052698,1.026612,-0.820389
4,1.463359,-0.600585,-0.430885,-1.080895,1.368408,1.435245,0.948180,-0.889516,0.842718,0.537721,-0.095216,0.997647,0.652978


### Transformamos los datos de PRUEBAS en los componentes anteriores

Para transformar los datos vino_prueba.csv estandarizados en las componentes anteriores, basta con crear la matriz 𝑈 cuyas columnas son los propios autovectore 𝑢𝑖 y hacer la multiplicación 𝑋·𝑈.

In [22]:
from sklearn.decomposition import PCA
import numpy as np

# Define num_componentes_80_percent based on previous analysis (5 components for 80% inertia)
num_componentes_80_percent = 5

# Crear una instancia de PCA con el número óptimo de componentes
pca_final = PCA(n_components=num_componentes_80_percent)

# Ajustar PCA a los datos de entrenamiento normalizados y transformarlos
X_train_ACP = pca_final.fit_transform(df_vino_entrenamiento_normalizado)

print(f"Dimensiones de los datos de entrenamiento después de PCA: {X_train_ACP.shape}")

Dimensiones de los datos de entrenamiento después de PCA: (142, 5)


Ahora ya sería multiplicar **df_autovectores_80_porciento** * **df_vino_prueba_normalizado**

In [23]:
# Multiplicamos las matrices de los datos de pruebas normalizados *
# matriz de los autovectores de los 5 primeros componentes (using pca_final.transform):
df_vinos_prueba_estandarizado = pca_final.transform(df_vino_prueba_normalizado)

#Esta multiplicacion de matrices representa la proyeccion manual de los componentes principales:
print(df_vinos_prueba_estandarizado.shape)

display(pd.DataFrame(df_vinos_prueba_estandarizado, columns=[f'PC{i+1}' for i in range(num_componentes_80_percent)]))

(36, 5)


,PC1,PC2,PC3,PC4,PC5
0,1.964615,-0.970458,-1.007767,3.478879,-0.891566
1,-0.649671,-2.636323,-1.317703,0.134402,-0.466142
2,3.113723,0.884861,-0.343076,0.767726,-0.627861
3,-0.223214,-2.860731,-0.905016,0.438346,-0.643266
4,2.730925,1.503240,-0.608390,0.304141,-0.539846
5,1.061430,-2.472155,-1.617986,0.019273,-0.100695
6,-0.092513,-1.965485,0.440210,0.954536,-0.476429
7,2.497761,1.886319,-0.415942,0.500581,-1.721827
8,-2.490902,2.099890,-0.805108,-0.393792,0.477916
9,3.484022,1.283084,-0.659252,-0.220301,0.065477


### **Clasificación Naïve-Bayes con PCA**

Ahora, utilizaremos los componentes principales transformados para entrenar y evaluar el modelo Naïve-Bayes.

In [26]:
#Necesitamos en formato matriz, no en dataset
X_test_estandarizado = df_vinos_prueba_estandarizado

#A pasar a la funcion de Clasificación retomamos los valores de entrenamiento estandarizados
y_train = df_vino_entrenamiento['class'].values

#Variables estandarizadas:
variables_ACP = [f'PC{i+1}' for i in range(num_componentes_80_percent)]  #que en nuestro caso equivale a poner CINCO como habíamos calculado

#Necesitamos que en el dataframe a pasar a la funcion de clasificación tenga el atributo objetvio CLASS
#junto con los datos
df_train_ACP = pd.DataFrame(X_train_ACP, columns=variables_ACP)
df_train_ACP['class'] = y_train

pred_ACP = naive_bayes(df_train_ACP, variables_ACP, 'class', X_test_estandarizado)

### **Evaluación del modelo Naïve-Bayes con ACP**

Evaluaremos el rendimiento del modelo Naïve-Bayes después de aplicar PCA y lo compararemos con el modelo original.

In [27]:
import pandas as pd
import numpy as np

# Extraer clases reales del dataset de prueba
y_real = df_vino_prueba[target].values

df_comp_ACP = pd.DataFrame({
    "Predicho_ACP": pred_ACP,
    "Real": y_real,
    "Correcto_ACP": pred_ACP == y_real
})
print(df_comp_ACP)

# Tabla con estadísticas (Matriz de confusión) para el modelo con PCA
print("\nMatriz de Confusión (PCA):")
print(pd.crosstab(df_comp_ACP['Real'], df_comp_ACP['Predicho_ACP']))

# Calcular y mostrar la precisión para cada clase con PCA
grupos = np.unique(y_real)
print("\nPrecisión por clase (PCA):")
for g in grupos:
    mask = (df_comp_ACP['Real'] == g)
    aciertos = df_comp_ACP.loc[mask, 'Correcto_ACP'].sum()
    total = mask.sum()
    acc = aciertos / total
    print(f"Clase {g}: {aciertos}/{total} aciertos ({acc:.2%})")

# Porcentaje de acierto total con PCA
accuracy_ACP = (pred_ACP == y_real).mean()
print(f"\nPorcentaje de aciertos TOTALES con PCA: {accuracy_ACP:.3f}")

# Comparación con la precisión del modelo original
print(f"\nPorcentaje de aciertos TOTALES sin PCA: {accuracy:.3f}")

   Predicho_ACP     Real  Correcto_ACP
0       Crianza  Crianza          True
1       Crianza  Crianza          True
2         Joven    Joven          True
3       Crianza  Crianza          True
4         Joven    Joven          True
5       Crianza  Crianza          True
6       Crianza  Crianza          True
7         Joven    Joven          True
8       Reserva  Reserva          True
9         Joven    Joven          True
10        Joven    Joven          True
11      Crianza  Crianza          True
12        Joven    Joven          True
13      Crianza  Crianza          True
14      Crianza  Crianza          True
15      Crianza  Crianza          True
16      Reserva  Crianza         False
17      Crianza  Crianza          True
18      Crianza  Crianza          True
19      Reserva  Reserva          True
20        Joven    Joven          True
21      Reserva  Reserva          True
22        Joven    Joven          True
23        Joven    Joven          True
24      Crianza  Crianza 